In [1]:
import pandas as pd

**Obtención de servicios DENUE por AGEB**

*Servicios DENUE 2020* 
Clave - Descripción de categoría
* 621610 - Servicios de enfermería a domicilio
* 623311 - Asilos y otras residencias del sector privado para el cuidado de ancianos
* 623312 - Asilos y otras residencias del sector público para el cuidado de ancianos

In [2]:
denue_2025 = pd.read_csv("../../data/raw/2025/denue2025.csv", encoding="latin-1")
denue_2025.info()

/tmp/ipykernel_45431/3243414725.py:1: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  denue_2025 = pd.read_csv("../../data/raw/2025/denue2025.csv", encoding="latin-1")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 460762 entries, 0 to 460761
Data columns (total 42 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          460762 non-null  int64  
 1   clee        460762 non-null  object 
 2   nom_estab   460762 non-null  object 
 3   raz_social  88476 non-null   object 
 4   codigo_act  460762 non-null  int64  
 5   nombre_act  460762 non-null  object 
 6   per_ocu     460762 non-null  object 
 7   tipo_vial   460429 non-null  object 
 8   nom_vial    460761 non-null  object 
 9   tipo_v_e_1  453606 non-null  object 
 10  nom_v_e_1   453595 non-null  object 
 11  tipo_v_e_2  453605 non-null  object 
 12  nom_v_e_2   453597 non-null  object 
 13  tipo_v_e_3  453604 non-null  object 
 14  nom_v_e_3   453422 non-null  object 
 15  numero_ext  435826 non-null  float64
 16  letra_ext   185208 non-null  object 
 17  edificio    100418 non-null  object 
 18  edificio_e  97456 non-null   object 
 19  nu

In [3]:
denue_2025.sample()

,id,clee,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,...,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
182427,779089,09005465111001711010000000U4,PERFUMERIA Y ARTICULOS DE FANTASIA,NaN,463215,Comercio al por menor de bisutería y accesorio...,0 a 5 personas,CALLE,NORTE 9,CALLE,...,Gustavo A. Madero ...,1383,5,NaN,NaN,NaN,Fijo,19.477531,-99.14499,2010-07


Lo que unicamente utilizaremos sera:
* Código de la clase de actividad
* Entidad federativa
* Municipio 
* Localidad
* Área geoestadística básica

In [4]:
denue_2025 = denue_2025[["codigo_act", "cve_ent", "entidad", "cve_mun", "municipio", "cve_loc", "localidad", "ageb"]]
denue_2025.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 460762 entries, 0 to 460761
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   codigo_act  460762 non-null  int64 
 1   cve_ent     460762 non-null  int64 
 2   entidad     460762 non-null  object
 3   cve_mun     460762 non-null  int64 
 4   municipio   460762 non-null  object
 5   cve_loc     460762 non-null  int64 
 6   localidad   460762 non-null  object
 7   ageb        460762 non-null  object
dtypes: int64(4), object(4)
memory usage: 28.1+ MB


Hay localidades no identificadas, pero esperemos a ver si dentro de nuestras actividades siguen habiendo.

In [5]:
denue_2025["cve_ent"] = denue_2025["cve_ent"].astype(str).str.zfill(2)
denue_2025["cve_mun"] = denue_2025["cve_mun"].astype(str).str.zfill(3)
denue_2025["cve_loc"] = denue_2025["cve_loc"].astype(str).str.zfill(4)
denue_2025["ageb"] = denue_2025["ageb"].str.zfill(4)
denue_2025.sample()

,codigo_act,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb
418971,811121,09,Ciudad de México,007,Iztapalapa,0001,Iztapalapa ...,1104


In [6]:
denue_2025["cvegeo"] = (denue_2025["cve_ent"] + denue_2025["cve_mun"] + denue_2025["cve_loc"] + denue_2025["ageb"])
denue_2025.sample()

,codigo_act,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb,cvegeo
255087,466212,09,Ciudad de México,012,Tlalpan,0001,Tlalpan ...,0905,0901200010905


In [7]:
serv_enf_dom = denue_2025[denue_2025["codigo_act"] == 621610]

In [8]:
serv_enf_dom = serv_enf_dom.groupby("cvegeo", as_index=False)["codigo_act"].count()
serv_enf_dom.rename(columns={"codigo_act":"cantidad_serv_enf_dom"}, inplace=True)
serv_enf_dom

,cvegeo,cantidad_serv_enf_dom
0,0900500011595,1
1,0900800010546,1
2,0901000011542,1
3,090100001231A,1
4,0901300010230,1
5,0901400010257,1
6,0901500011341,1
7,0901600010660,1


In [9]:
asilos = denue_2025[denue_2025["codigo_act"].isin([623311, 623312])]
asilos

,codigo_act,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb,cvegeo
304632,623312,09,Ciudad de México,014,Benito Juárez,0001,Benito Juárez ...,0384,0901400010384
304639,623312,09,Ciudad de México,003,Coyoacán,0001,Coyoacán ...,0094,0900300010094
304644,623312,09,Ciudad de México,014,Benito Juárez,0001,Benito Juárez ...,0204,0901400010204
305013,623311,09,Ciudad de México,016,Miguel Hidalgo,0001,Miguel Hidalgo ...,0603,0901600010603
305156,623311,09,Ciudad de México,014,Benito Juárez,0001,Benito Juárez ...,0632,0901400010632
...,...,...,...,...,...,...,...,...,...
326521,623311,09,Ciudad de México,008,La Magdalena Contreras,0001,La Magdalena Contreras ...,0052,0900800010052
326606,623311,09,Ciudad de México,014,Benito Juárez,0001,Benito Juárez ...,0261,0901400010261
326688,623311,09,Ciudad de México,008,La Magdalena Contreras,0001,La Magdalena Contreras ...,0283,0900800010283
326815,623311,09,Ciudad de México,002,Azcapotzalco,0001,Azcapotzalco ...,0699,0900200010699


In [10]:
asilos = asilos.groupby("cvegeo", as_index=False)["codigo_act"].count()
asilos.rename(columns={"codigo_act":"cantidad_asilos"}, inplace=True)
asilos

,cvegeo,cantidad_asilos
0,0900200010345,2
1,0900200010699,1
2,0900200010792,1
3,0900300010094,1
4,0900300010111,2
...,...,...
92,0901600011052,1
93,0901600011175,1
94,0901600011245,1
95,0901700010920,1


In [11]:
m = serv_enf_dom.merge(asilos, on="cvegeo", how="outer")

In [12]:
m.fillna(0)

,cvegeo,cantidad_serv_enf_dom,cantidad_asilos
0,0900200010345,0.0,2.0
1,0900200010699,0.0,1.0
2,0900200010792,0.0,1.0
3,0900300010094,0.0,1.0
4,0900300010111,0.0,2.0
...,...,...,...
99,0901600011052,0.0,1.0
100,0901600011175,0.0,1.0
101,0901600011245,0.0,1.0
102,0901700010920,0.0,1.0


In [13]:
m.to_csv("../../data/processed/datos_denue_2025.csv")